# Collagen Organization in Breast Carcinoma Stroma
## Quantifying Tumor-Associated Collagen Signatures (TACS) via Structure Tensor Coherence
**DigitalSreeni — AI in Digital Pathology Series**

---

### Clinical background

The organization of collagen fibers in the tumor microenvironment is not just structural scaffolding — it actively directs cancer cell invasion.

**Tumor-Associated Collagen Signatures (TACS)** is a framework introduced by Keely et al. that classifies collagen organization relative to a tumor boundary:

| TACS grade | Collagen pattern | Clinical meaning |
|---|---|---|
| TACS-1 | Dense, locally straightened collagen near tumor | Early indicator of tumor presence |
| TACS-2 | Collagen aligned **parallel** to tumor boundary | Tumor is constrained — less aggressive |
| TACS-3 | Collagen aligned **perpendicular** to tumor boundary | Active invasion — strongly associated with poor prognosis |

TACS-3 is the most clinically significant: perpendicular collagen fibers act as invasion highways, guiding tumor cells away from the primary mass. Patients with high TACS-3 burden have significantly worse survival in breast cancer.

### The measurement problem

Manually scoring TACS requires a trained pathologist and is subjective, slow, and not reproducible at scale. Automated methods need to quantify:
1. **How organized** the collagen is locally (coherence)
2. **What direction** it runs relative to the tumor boundary (orientation)

The structure tensor coherence map computes both from standard H&E images — no special staining, no deep learning model, no training data.

### What this notebook demonstrates

- Load an H&E image of invasive breast carcinoma
- Compute the structure tensor coherence and orientation maps
- Quantify the difference between tumor nests (disorganized stroma) and aligned collagen stroma
- Segment low-coherence zones and extract per-region metrics
- Visualize orientation distributions per tissue zone


## 0 — Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Uncomment if running locally:
# !pip install scikit-image scipy numpy matplotlib pillow

import numpy as np
from PIL import Image

from scipy.ndimage import (gaussian_filter, label,
                           binary_opening, binary_closing)
from skimage.measure import regionprops
from skimage.filters import threshold_otsu

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Libraries imported OK")


## 1 — Load the H&E image

Upload your H&E image via the Colab Files panel and set `IMG_PATH`.

This image contains three tissue zones from left to right:
- **Left ~70%** — tumor nests + disorganized desmoplastic stroma (including a transitional mid-stroma zone where fibers curve around a vessel)
- **Right ~30%** — dense aligned collagen stroma (long parallel fibers, elongated fibroblast nuclei)


In [ ]:
IMG_PATH = "/content/drive/MyDrive/ColabNotebooks/data/HE_fibrous_tissue2.png"
img_pil  = Image.open(IMG_PATH)
img_rgb  = np.array(img_pil.convert("RGB"))
img_gray = np.array(img_pil.convert("L"), dtype=float)
H, W = img_gray.shape

print(f"Image size: {W} x {H} pixels")
print(f"Grayscale range: {img_gray.min():.0f} – {img_gray.max():.0f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_rgb)
ax.set_title("H&E — Invasive breast carcinoma with stromal collagen", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()


## 2 — Identify tissue zones

This image has a continuous gradient of collagen organization rather than two flat zones — which is exactly what real tissue looks like. We define two zones for quantitative comparison:

- **Tumor / disorganized zone** — left 70% of the image (tumor nests, disorganized stroma, transitional mid-stroma)
- **Aligned fiber zone** — right 30% of the image (dense parallel collagen)

The boundary at 70% is approximate and chosen to capture the clearly aligned fibers on the right while including the full extent of disorganized and transitional stroma on the left.


In [ ]:
ZONE_SPLIT = int(W * 0.7)   # column index separating disorganized from aligned zone

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(img_rgb)
axes[0].set_title("Original H&E", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_rgb)
rect_tumor = plt.Rectangle((0, 0), ZONE_SPLIT, H,
                            linewidth=0, facecolor="red", alpha=0.12)
rect_fiber = plt.Rectangle((ZONE_SPLIT, 0), W - ZONE_SPLIT, H,
                            linewidth=0, facecolor="blue", alpha=0.12)
axes[1].add_patch(rect_tumor)
axes[1].add_patch(rect_fiber)
axes[1].axvline(ZONE_SPLIT, color="white", lw=2, linestyle="--")
axes[1].text(ZONE_SPLIT // 2, 60,
             "Tumor nests\n+ disorganized stroma", color="white",
             fontsize=11, fontweight="bold", ha="center",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="red", alpha=0.6))
axes[1].text(ZONE_SPLIT + (W - ZONE_SPLIT) // 2, 60,
             "Aligned\ncollagen stroma", color="white",
             fontsize=11, fontweight="bold", ha="center",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="blue", alpha=0.6))
axes[1].set_title("Annotated tissue zones", fontsize=12, fontweight="bold")
axes[1].axis("off")

fig.suptitle("Step 1 — Tissue zone identification", fontsize=13)
plt.tight_layout()
plt.show()


## 3 — Compute image gradients

Gaussian derivative filtering gives us the partial derivatives Iₓ (horizontal) and I_y (vertical).

**Parameter choice for H&E:**
- `SIGMA = 2` — collagen fibers in H&E at 20× magnification are typically 3–8 px wide. Sigma=2 resolves individual fiber boundaries without blurring across them.
- For higher magnification (40×), increase to 3–4.


In [ ]:
SIGMA = 2   # gradient smoothing — matched to collagen fiber width

Ix = gaussian_filter(img_gray, sigma=SIGMA, order=[0, 1])
Iy = gaussian_filter(img_gray, sigma=SIGMA, order=[1, 0])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
m = np.percentile(np.abs(Ix), 99)
axes[0].imshow(Ix, cmap="RdBu_r", vmin=-m, vmax=m)
axes[0].set_title("Gradient Ix  (horizontal)", fontsize=11); axes[0].axis("off")
m = np.percentile(np.abs(Iy), 99)
axes[1].imshow(Iy, cmap="RdBu_r", vmin=-m, vmax=m)
axes[1].set_title("Gradient Iy  (vertical)", fontsize=11); axes[1].axis("off")
fig.suptitle("Step 2 — Image gradients", fontsize=13)
plt.tight_layout(); plt.show()

grad_mag = np.sqrt(Ix**2 + Iy**2)
print("Gradient magnitude by zone:")
print(f"  Tumor / disorganized (left 70%):  {grad_mag[:, :ZONE_SPLIT].mean():.2f}")
print(f"  Aligned fiber (right 30%):        {grad_mag[:, ZONE_SPLIT:].mean():.2f}")


## 4 — Build the structure tensor

At each pixel, form the outer product of the gradient vector, then smooth with Gaussian Gᵨ:

$$J = G_\rho * \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

**`RHO` for collagen fibers:**
Set `RHO` to span 1–2 fiber widths so the tensor integrates over enough fiber edges to detect the local orientation reliably. At 20× magnification with ~10–20 px fiber spacing, `RHO = 20` works well.


In [ ]:
RHO = 20   # tensor neighbourhood — should span 1-2 collagen fiber cycles

Jxx = gaussian_filter(Ix * Ix, sigma=RHO)
Jxy = gaussian_filter(Ix * Iy, sigma=RHO)
Jyy = gaussian_filter(Iy * Iy, sigma=RHO)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, data, title in zip(axes, [Jxx, Jxy, Jyy],
                           ["Jxx  (Ix²)", "Jxy  (Ix·Iy)", "Jyy  (Iy²)"]):
    ax.imshow(data, cmap="inferno")
    ax.set_title(f"Tensor: {title}", fontsize=11); ax.axis("off")
fig.suptitle("Step 3 — Structure tensor components (after neighbourhood smoothing)", fontsize=13)
plt.tight_layout(); plt.show()


## 5 — Coherence and orientation maps

$$C = \frac{\sqrt{(J_{xx}-J_{yy})^2 + 4J_{xy}^2}}{J_{xx}+J_{yy}} \qquad \theta = \frac{1}{2}\arctan\left(\frac{2J_{xy}}{J_{xx}-J_{yy}}\right)$$

**What to expect in this image:**
- Aligned collagen stroma (right): all fiber edges point roughly the same direction → coherence near 0.8–0.9
- Tumor nests / disorganized stroma (left): collagen wraps around cell clusters in every direction → coherence near 0.2–0.3
- Transitional mid-stroma: coherence varies 0.3–0.6 depending on local fiber curvature
- The coherence map produces a continuous map of collagen organization — not a binary output


In [ ]:
diff        = Jxx - Jyy
coherence   = np.sqrt(diff**2 + 4 * Jxy**2) / (Jxx + Jyy + 1e-10)
orientation = 0.5 * np.arctan2(2 * Jxy, Jxx - Jyy)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(coherence, cmap="magma", vmin=0, vmax=1)
axes[0].set_title(
    "Coherence map\nBright = aligned collagen  |  Dark = tumor / disorganized stroma",
    fontsize=11, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, label="Coherence (C)")

im2 = axes[1].imshow(np.degrees(orientation), cmap="hsv", vmin=-90, vmax=90)
axes[1].set_title(
    "Fiber orientation map\nColour = dominant collagen direction at each pixel",
    fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, label="Degrees (θ)")

fig.suptitle("Step 4 — Coherence and orientation maps", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

print("Coherence by zone:")
print(f"  Tumor / disorganized (left 70%):  {coherence[:, :ZONE_SPLIT].mean():.3f} ± {coherence[:, :ZONE_SPLIT].std():.3f}")
print(f"  Aligned fiber (right 30%):        {coherence[:, ZONE_SPLIT:].mean():.3f} ± {coherence[:, ZONE_SPLIT:].std():.3f}")
print(f"  Ratio (fiber / tumor):            {coherence[:, ZONE_SPLIT:].mean() / coherence[:, :ZONE_SPLIT].mean():.2f}×")


## 6 — Coherence profile across the image

Plotting mean coherence per column reveals the continuous gradient — from near zero at the tumor core, through partial alignment in the transitional stroma, to high coherence in the dense fiber zone.

This gradient is what a real WSI coherence heatmap looks like. On a whole slide you would compute this across a grid of patches tiled across the biopsy, and the coherence map becomes a spatial heatmap of stromal organization.


In [ ]:
col_profile = coherence.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(W), col_profile, color="#FF6B6B", lw=1.5, label="Mean coherence per column")

otsu_thresh = threshold_otsu(coherence)
ax.axhline(otsu_thresh, color="black", lw=1.5, linestyle="--",
           label=f"Otsu threshold ({otsu_thresh:.2f})")
ax.axvline(ZONE_SPLIT, color="grey", lw=1.2, linestyle=":",
           label=f"Zone boundary (col {ZONE_SPLIT})")

ax.fill_between(range(W), col_profile, otsu_thresh,
                where=col_profile < otsu_thresh,
                alpha=0.25, color="red", label="Low coherence")
ax.fill_between(range(W), col_profile, otsu_thresh,
                where=col_profile >= otsu_thresh,
                alpha=0.25, color="blue", label="High coherence")

ax.set_xlabel("Column  (left = tumor,  right = aligned fiber)", fontsize=12)
ax.set_ylabel("Mean coherence", fontsize=12)
ax.set_title("Coherence profile across the image — continuous gradient, not a step function",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 7 — Coherence distribution: the bimodal signature

Even in this more complex image the histogram is bimodal — the two peaks correspond to disorganized and aligned tissue. The valley between them is where Otsu's method places the threshold automatically. No manual tuning needed.


In [ ]:
otsu_thresh = threshold_otsu(coherence)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(coherence[:, :ZONE_SPLIT].ravel(), bins=80, alpha=0.75, color="#FF5050",
        label="Tumor / disorganized stroma (left 70%)", density=True)
ax.hist(coherence[:, ZONE_SPLIT:].ravel(), bins=80, alpha=0.75, color="#2196F3",
        label="Aligned collagen stroma (right 30%)", density=True)
ax.axvline(otsu_thresh, color="black", lw=2, linestyle="--",
           label=f"Otsu threshold ({otsu_thresh:.2f})")
ax.set_xlabel("Coherence value  (C)", fontsize=13)
ax.set_ylabel("Density", fontsize=13)
ax.set_title("Coherence distribution by tissue zone — bimodal even in complex tissue",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Otsu threshold: {otsu_thresh:.3f}")
print(f"Pixels above threshold (organized):    {100*(coherence > otsu_thresh).mean():.1f}%")
print(f"Pixels below threshold (disorganized): {100*(coherence <= otsu_thresh).mean():.1f}%")


## 8 — Segment disorganized zones

Threshold below Otsu → binary mask → morphological cleanup → connected component labeling.

Low-coherence regions capture tumor nests, the disorganized desmoplastic stroma immediately surrounding them, and transitional zones — all the tissue that would score low on a TACS assessment.


In [ ]:
otsu_thresh = threshold_otsu(coherence)

raw_mask = coherence < otsu_thresh
cleaned  = binary_opening(raw_mask, iterations=3)
cleaned  = binary_closing(cleaned,  iterations=3)

labeled, n_regions = label(cleaned)
props_all = regionprops(labeled)
props_big = sorted([p for p in props_all if p.area > 3000],
                   key=lambda p: p.area, reverse=True)

print(f"Total connected regions: {n_regions}")
print(f"Regions with area > 3000px: {len(props_big)}")
print()
print(f"{'Region':>8} {'Area (px)':>12} {'Equiv. radius':>15} {'Eccentricity':>14}")
print("-" * 55)
for p in props_big:
    r = np.sqrt(p.area / np.pi)
    print(f"{p.label:>8} {p.area:>12.0f} {r:>15.1f} {p.eccentricity:>14.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(coherence, cmap="magma")
axes[0].set_title("Coherence map"); axes[0].axis("off")
axes[1].imshow(raw_mask, cmap="gray")
axes[1].set_title(f"Threshold  (Otsu = {otsu_thresh:.2f})"); axes[1].axis("off")
axes[2].imshow(cleaned, cmap="gray")
axes[2].set_title("After morphological cleanup"); axes[2].axis("off")
fig.suptitle("Step 5 — Segmenting disorganized stroma", fontsize=13)
plt.tight_layout(); plt.show()


## 9 — Overlay: disorganized zones on original H&E

In [ ]:
from skimage.segmentation import find_boundaries

boundary = find_boundaries(cleaned, mode="outer")

overlay = img_rgb.copy().astype(float)
overlay[cleaned]  = overlay[cleaned]  * 0.2 + np.array([220, 70, 70])  * 0.8
overlay[boundary] = [255, 220, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb)
axes[0].set_title("Original H&E", fontsize=12, fontweight="bold"); axes[0].axis("off")
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title("Disorganized stroma segmented\n(red = low coherence,  yellow = boundary)",
                  fontsize=12, fontweight="bold"); axes[1].axis("off")

fig.suptitle("Step 6 — Disorganized stroma overlay", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 10 — Orientation rose plots per tissue zone

The orientation map gives a direction at every pixel. Plotting it as a **rose diagram** (polar histogram) reveals:
- Tumor zone: flat distribution — no preferred direction
- Fiber zone: one dominant peak — fibers all run at approximately the same angle

**Important note on circular statistics:** because orientation is axial (a fiber at +34° is physically identical to one at −134°), we double the angles before plotting to handle the 180° ambiguity. The **resultant vector length R** is the correct alignment metric — not the linear standard deviation, which gives misleadingly large values on axial data.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), subplot_kw=dict(projection="polar"))

zone_data = [
    (orientation[:, :ZONE_SPLIT], "Tumor zone — disorganized stroma", "#FF5050"),
    (orientation[:, ZONE_SPLIT:], "Fiber zone — aligned collagen",     "#2196F3"),
]

for ax, (angles, title, color) in zip(axes, zone_data):
    doubled = (2 * angles.ravel()) % (2 * np.pi)
    ax.hist(doubled, bins=36, color=color, alpha=0.85, density=True)
    ax.set_title(title, pad=15, fontsize=11, fontweight="bold")
    ax.set_theta_zero_location("E")

    sin_mean = np.sin(doubled).mean()
    cos_mean = np.cos(doubled).mean()
    R = np.sqrt(sin_mean**2 + cos_mean**2)
    circ_std = np.degrees(np.sqrt(-2 * np.log(R + 1e-10))) / 2
    ax.set_xlabel(f"Resultant vector R = {R:.3f}  |  Circ. std = {circ_std:.1f}°",
                  labelpad=15, fontsize=10)

fig.suptitle("Step 7 — Fiber orientation distribution by tissue zone", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 11 — Quantitative metrics summary

These are the numbers that would go into a paper or pathology report.

The **Coherence Anisotropy Index (CAI)** is the mean coherence per region. In published TACS literature, CAI computed from SHG images correlates with invasion grade and patient survival. The structure tensor computed here from H&E gives equivalent information without SHG.


In [ ]:
otsu_thresh = threshold_otsu(coherence)

print("=" * 58)
print("  TACS QUANTIFICATION REPORT")
print("=" * 58)

zone_labels = ["Tumor / disorganized stroma  (left 70%)",
               "Aligned collagen stroma       (right 30%)"]
zone_slices = [np.s_[:, :ZONE_SPLIT], np.s_[:, ZONE_SPLIT:]]

for zname, zslice in zip(zone_labels, zone_slices):
    coh_zone = coherence[zslice]
    ori_zone = orientation[zslice]
    doubled  = (2 * ori_zone.ravel()) % (2 * np.pi)
    R        = np.sqrt(np.sin(doubled).mean()**2 + np.cos(doubled).mean()**2)
    circ_std = np.degrees(np.sqrt(-2 * np.log(R + 1e-10))) / 2
    dom_deg  = np.degrees(np.arctan2(np.sin(doubled).mean(),
                                     np.cos(doubled).mean()) / 2)
    print(f"\n  {zname}")
    print(f"    CAI (mean coherence):             {coh_zone.mean():.3f} ± {coh_zone.std():.3f}")
    print(f"    Resultant vector R:               {R:.3f}  (0=random, 1=perfect)")
    print(f"    Circular std of orientation:      {circ_std:.1f}°")
    print(f"    Dominant fiber angle:             {dom_deg:.1f}°")

ratio = coherence[:, ZONE_SPLIT:].mean() / coherence[:, :ZONE_SPLIT].mean()
print(f"\n  Overall")
print(f"    CAI ratio (fiber / tumor):        {ratio:.2f}×")
print(f"    Disorganized area fraction:       {100*(coherence <= otsu_thresh).mean():.1f}%")
print(f"    Organized area fraction:          {100*(coherence > otsu_thresh).mean():.1f}%")
print("=" * 58)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

zones     = ["Tumor /\ndisorganized", "Aligned\ncollagen"]
coh_means = [coherence[:, :ZONE_SPLIT].mean(), coherence[:, ZONE_SPLIT:].mean()]
coh_stds  = [coherence[:, :ZONE_SPLIT].std(),  coherence[:, ZONE_SPLIT:].std()]

bars = axes[0].bar(zones, coh_means, yerr=coh_stds, capsize=7,
                   color=["#FF5050", "#2196F3"],
                   edgecolor="white", linewidth=1.5, width=0.5)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Coherence Anisotropy Index (CAI)", fontsize=11)
axes[0].set_title("CAI by tissue zone", fontsize=12, fontweight="bold")
for bar, val in zip(bars, coh_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.04, f"{val:.2f}",
                ha="center", fontsize=14, fontweight="bold")
axes[0].grid(True, axis="y", alpha=0.3)

otsu_thresh = threshold_otsu(coherence)
sizes  = [(coherence > otsu_thresh).sum(), (coherence <= otsu_thresh).sum()]
axes[1].pie(sizes,
            labels=["High coherence\n(organized collagen)",
                    "Low coherence\n(disorganized / tumor)"],
            colors=["#2196F3", "#FF5050"], autopct="%1.1f%%",
            startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Tissue composition by orientation order",
                  fontsize=12, fontweight="bold")

plt.suptitle("Step 8 — Quantitative TACS metrics", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 12 — Clinical interpretation

### Reading the numbers

| Metric | This image | Interpretation |
|---|---|---|
| CAI — tumor zone | ~0.23 | Highly disorganized — active desmoplasia |
| CAI — fiber zone | ~0.66 | Strongly aligned — TACS-2/3 pattern |
| CAI ratio | ~2.86× | Large contrast — reliable automated separation |
| Fiber alignment R | ~0.86 | Strong directional coherence in stroma |
| Disorganized fraction | ~70% | Majority of field is tumor-influenced stroma |

### Why the transitional zone matters

Unlike the idealized two-zone case, this image has a continuous gradient of coherence from the tumor core outward. That gradient is biologically meaningful — it maps the spatial extent of tumor influence on the stromal microenvironment. High-grade invasive tumors tend to show disorganization extending further into the stroma.

### TACS classification from coherence

To classify TACS-3 (perpendicular invasion fibers) automatically:
1. Detect the tumor-stroma boundary from the segmentation mask edge
2. Compute the boundary normal vector at each point
3. Compare local fiber orientation θ to that normal — if within ~15°, classify as TACS-3

The structure tensor gives you step 3 for free. On a whole slide image you would tile the biopsy into patches, compute CAI and dominant orientation per patch, then map the results spatially — producing a TACS heatmap across the entire tumor boundary without any manual annotation.

### Key references

- Provenzano et al. (2006) BMC Medicine — TACS framework, collagen reorganization and invasion
- Schedin & Keely (2011) Cold Spring Harb Perspect Biol — mammary ECM remodeling and tumor progression
- Bredfeldt et al. (2014) Journal of Biomedical Optics — automated collagen fiber quantification from SHG images
- Avila & Bueno (2015) Applied Optics — structure tensor for collagen organization quantification in SHG images
